In [1]:
import math
class TicTacToe:
    def __init__(self):
        self.board = [" " for _ in range(9)]

    def print_board(self):
        print()
        for i in range(3):
            row = self.board[i * 3:(i + 1) * 3]
            print(" " + " | ".join(row))
            if i < 2:
                print("---+---+---")
        print()

    def print_positions(self):
        print()
        print("Position numbers:")
        for i in range(3):
            row = [str(i * 3 + j + 1) for j in range(3)]
            print(" " + " | ".join(row))
            if i < 2:
                print("---+---+---")
        print()

    def available_moves(self):
        moves = []
        for i in range(9):
            if self.board[i] == " ":
                moves.append(i)
        return moves

    def make_move(self, index, player):
        if 0 <= index < 9 and self.board[index] == " ":
            self.board[index] = player
            return True
        return False

    def undo_move(self, index):
        self.board[index] = " "

    def is_winner(self, player):
        win_patterns = [
            [0, 1, 2],
            [3, 4, 5],
            [6, 7, 8],
            [0, 3, 6],
            [1, 4, 7],
            [2, 5, 8],
            [0, 4, 8],
            [2, 4, 6]
        ]

        for pattern in win_patterns:
            if (
                self.board[pattern[0]] == player and
                self.board[pattern[1]] == player and
                self.board[pattern[2]] == player
            ):
                return True
        return False

    def is_draw(self):
        return " " not in self.board and not self.is_winner("X") and not self.is_winner("O")

    def game_over(self):
        return self.is_winner("X") or self.is_winner("O") or self.is_draw()

    def evaluate(self, ai_player, human_player, depth):
        if self.is_winner(ai_player):
            return 10 - depth
        if self.is_winner(human_player):
            return depth - 10
        return 0

    def minimax(self, depth, is_maximizing, ai_player, human_player, alpha, beta, max_depth):
        score = self.evaluate(ai_player, human_player, depth)

        if score != 0:
            return score

        if self.is_draw():
            return 0

        if max_depth != -1 and depth >= max_depth:
            return 0

        if is_maximizing:
            best_score = -math.inf
            for move in self.available_moves():
                self.make_move(move, ai_player)
                value = self.minimax(depth + 1, False, ai_player, human_player, alpha, beta, max_depth)
                self.undo_move(move)

                if value > best_score:
                    best_score = value

                if best_score > alpha:
                    alpha = best_score

                if beta <= alpha:
                    break

            return best_score
        else:
            best_score = math.inf
            for move in self.available_moves():
                self.make_move(move, human_player)
                value = self.minimax(depth + 1, True, ai_player, human_player, alpha, beta, max_depth)
                self.undo_move(move)

                if value < best_score:
                    best_score = value

                if best_score < beta:
                    beta = best_score

                if beta <= alpha:
                    break

            return best_score

    def best_move(self, ai_player, human_player, max_depth):
        best_score = -math.inf
        best_move_index = None

        for move in self.available_moves():
            self.make_move(move, ai_player)
            score = self.minimax(0, False, ai_player, human_player, -math.inf, math.inf, max_depth)
            self.undo_move(move)

            if score > best_score:
                best_score = score
                best_move_index = move

        return best_move_index


def get_depth_input(player_name):
    while True:
        value = input(f"Enter search depth for {player_name} (-1 for full search): ").strip()
        try:
            depth = int(value)
            if depth >= -1:
                return depth
            else:
                print("Please enter -1 or a non-negative integer.")
        except ValueError:
            print("Invalid input. Please enter an integer.")


def human_turn(game, player):
    while True:
        move = input(f"Player {player}, enter your move (1-9): ").strip()
        try:
            move = int(move) - 1
            if game.make_move(move, player):
                break
            else:
                print("Invalid move. Try again.")
        except ValueError:
            print("Please enter a number from 1 to 9.")


def ai_turn(game, ai_player, opponent_player, depth):
    move = game.best_move(ai_player, opponent_player, depth)
    if move is not None:
        game.make_move(move, ai_player)
        print(f"Computer ({ai_player}) chooses position {move + 1}")


def play_human_vs_computer():
    game = TicTacToe()
    game.print_positions()

    while True:
        human = input("Choose your symbol (X/O): ").strip().upper()
        if human in ["X", "O"]:
            break
        print("Please choose X or O.")

    computer = "O" if human == "X" else "X"
    depth = get_depth_input("Computer")

    current_player = "X"

    while not game.game_over():
        print(f"Current turn: {current_player}")

        if current_player == human:
            human_turn(game, human)
        else:
            ai_turn(game, computer, human, depth)

        game.print_board()

        if game.is_winner(current_player):
            if current_player == human:
                print("Human wins!")
            else:
                print("Computer wins!")
            return

        if game.is_draw():
            print("Game ended in a draw.")
            return

        current_player = "O" if current_player == "X" else "X"


def play_computer_vs_computer():
    game = TicTacToe()
    game.print_positions()

    depth_x = get_depth_input("Computer X")
    depth_o = get_depth_input("Computer O")

    current_player = "X"

    while not game.game_over():
        print(f"Current turn: {current_player}")

        if current_player == "X":
            ai_turn(game, "X", "O", depth_x)
        else:
            ai_turn(game, "O", "X", depth_o)

        game.print_board()

        if game.is_winner(current_player):
            print(f"Computer {current_player} wins!")
            return

        if game.is_draw():
            print("Game ended in a draw.")
            return

        current_player = "O" if current_player == "X" else "X"


def main():
    print("Tic Tac Toe with Minimax and Alpha-Beta Pruning")
    print("1. Human vs Computer")
    print("2. Computer vs Computer")

    while True:
        choice = input("Choose game mode (1 or 2): ").strip()
        if choice == "1":
            play_human_vs_computer()
            break
        elif choice == "2":
            play_computer_vs_computer()
            break
        else:
            print("Invalid choice. Please enter 1 or 2.")


if __name__ == "__main__":
    main()


Tic Tac Toe with Minimax and Alpha-Beta Pruning
1. Human vs Computer
2. Computer vs Computer
Choose game mode (1 or 2): 1

Position numbers:
 1 | 2 | 3
---+---+---
 4 | 5 | 6
---+---+---
 7 | 8 | 9

Choose your symbol (X/O): X
Enter search depth for Computer (-1 for full search): 4
Current turn: X
Player X, enter your move (1-9): 1

 X |   |  
---+---+---
   |   |  
---+---+---
   |   |  

Current turn: O
Computer (O) chooses position 2

 X | O |  
---+---+---
   |   |  
---+---+---
   |   |  

Current turn: X
Player X, enter your move (1-9): 5

 X | O |  
---+---+---
   | X |  
---+---+---
   |   |  

Current turn: O
Computer (O) chooses position 9

 X | O |  
---+---+---
   | X |  
---+---+---
   |   | O

Current turn: X
Player X, enter your move (1-9): 4

 X | O |  
---+---+---
 X | X |  
---+---+---
   |   | O

Current turn: O
Computer (O) chooses position 3

 X | O | O
---+---+---
 X | X |  
---+---+---
   |   | O

Current turn: X
Player X, enter your move (1-9): 7

 X | O | O
---